# Part 4 — Verify Training Outputs

Checks all 24 prediction zip files and generates correct `paths_config.yaml`.

**Fixes applied:**
- Level models: uses `starts_with='train'` or `starts_with='val'` to distinguish files
- Diff models: uses `'_train.'` or `'_val.'` in filename to distinguish train vs val

## ① Mount Drive

In [1]:
import os, yaml
from pathlib import Path

# Detect project root
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

ROOT      = str(PROJECT_ROOT) + os.sep
PRED_DIR  = str(PROJECT_ROOT / 'data' / 'predictions') + os.sep

print(f'Project root: {PROJECT_ROOT}')
print(f'Predictions:  {PRED_DIR}')

Project root: /home/iankuzuma/claude_code/demand_modeling/men-8-subcat-split-proper-embedding/fashion-sneakers
Predictions:  /home/iankuzuma/claude_code/demand_modeling/men-8-subcat-split-proper-embedding/fashion-sneakers/data/predictions/


## ② Check All 24 Files Exist

In [2]:
print('=== OUTPUT FILES ===\n')
total = 0
for subdir in ['txt/time_indipendent', 'txt/lag1',
               'txtimg/time_indipendent', 'txtimg/lag1']:
    path  = PRED_DIR + subdir
    files = sorted([f for f in os.listdir(path) if f.endswith('.zip')])
    print(f'{subdir}/ ({len(files)} files):')
    for f in files:
        mb = os.path.getsize(path + '/' + f) / 1e6
        print(f'  {f}  ({mb:.1f} MB)')
    print()
    total += len(files)

print(f'Total : {total} zip files  (expected: 24)')
if total == 24:
    print('\n✅ All 24 files produced')
else:
    print(f'\n⚠️  {24 - total} files missing')

=== OUTPUT FILES ===

txt/time_indipendent/ (6 files):
  pred_diff_shoes-diff-model-txt_2026-05-31-embdim=768_lag1-epoch=015-val_combined_loss=0.3187_txt_invariant_train.zip  (0.1 MB)
  pred_diff_shoes-diff-model-txt_2026-05-31-embdim=768_lag1-epoch=015-val_combined_loss=0.3187_txt_invariant_val.zip  (0.1 MB)
  train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip  (0.2 MB)
  train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip  (0.1 MB)
  val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip  (0.2 MB)
  val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip  (0.1 MB)

txt/lag1/ (6 files):
  pred_diff_shoes-diff-model-txt_2026-05-31-embdim=768_lag1-epoch=015-val_combined_loss=0.2999_

## ③ Generate paths_config.yaml

**Two separate fix strategies:**
- **Level models** (`train_pred_model` / `val_pred_model`): use `starts_with` to match file prefix
- **Diff models** (`pred_diff_..._train.zip` / `pred_diff_..._val.zip`): use `_train.` or `_val.` suffix in `must_contain`

In [3]:
def find_file(subdir, must_contain, must_not_contain=None, starts_with=None):
    path  = PRED_DIR + subdir
    files = [f for f in os.listdir(path) if f.endswith('.zip')]
    for f in sorted(files):
        if starts_with and not f.startswith(starts_with):
            continue
        if all(p in f for p in must_contain):
            if must_not_contain is None or not any(p in f for p in must_not_contain):
                return f'../data/predictions/{subdir}/{f}'
    return None

config = {
    'txt': {
        'time_independent': {
            128: {
                'train': find_file('txt/time_indipendent', ['dim=128'], ['diff'], starts_with='train'),
                'val':   find_file('txt/time_indipendent', ['dim=128'], ['diff'], starts_with='val'),
            },
            256: {
                'train': find_file('txt/time_indipendent', ['dim=256'], ['diff'], starts_with='train'),
                'val':   find_file('txt/time_indipendent', ['dim=256'], ['diff'], starts_with='val'),
            },
        },
        'lag1': {
            128: {
                'train': find_file('txt/lag1', ['dim=128'], ['diff'], starts_with='train'),
                'val':   find_file('txt/lag1', ['dim=128'], ['diff'], starts_with='val'),
            },
            256: {
                'train': find_file('txt/lag1', ['dim=256'], ['diff'], starts_with='train'),
                'val':   find_file('txt/lag1', ['dim=256'], ['diff'], starts_with='val'),
            },
        },
    },
    'txtimg': {
        'time_independent': {
            128: {
                'train': find_file('txtimg/time_indipendent', ['dim=128'], ['diff'], starts_with='train'),
                'val':   find_file('txtimg/time_indipendent', ['dim=128'], ['diff'], starts_with='val'),
            },
            256: {
                'train': find_file('txtimg/time_indipendent', ['dim=256'], ['diff'], starts_with='train'),
                'val':   find_file('txtimg/time_indipendent', ['dim=256'], ['diff'], starts_with='val'),
            },
        },
        'lag1': {
            128: {
                'train': find_file('txtimg/lag1', ['dim=128'], ['diff'], starts_with='train'),
                'val':   find_file('txtimg/lag1', ['dim=128'], ['diff'], starts_with='val'),
            },
            256: {
                'train': find_file('txtimg/lag1', ['dim=256'], ['diff'], starts_with='train'),
                'val':   find_file('txtimg/lag1', ['dim=256'], ['diff'], starts_with='val'),
            },
        },
    },
    # Diff models use _train. and _val. suffix to distinguish
    'diff_txt': {
        'time_independent': {
            'train': find_file('txt/time_indipendent', ['invariant', '_train.'], starts_with='pred'),
            'val':   find_file('txt/time_indipendent', ['invariant', '_val.'],   starts_with='pred'),
        },
        'lag1': {
            'train': find_file('txt/lag1', ['lag1', '_train.'], starts_with='pred'),
            'val':   find_file('txt/lag1', ['lag1', '_val.'],   starts_with='pred'),
        },
    },
    'diff_txtimg': {
        'time_independent': {
            'train': find_file('txtimg/time_indipendent', ['invariant', '_train.'], starts_with='pred'),
            'val':   find_file('txtimg/time_indipendent', ['invariant', '_val.'],   starts_with='pred'),
        },
        'lag1': {
            'train': find_file('txtimg/lag1', ['lag1', '_train.'], starts_with='pred'),
            'val':   find_file('txtimg/lag1', ['lag1', '_val.'],   starts_with='pred'),
        },
    },
}

# Verify all paths
print('=== Verifying all paths ===')
missing = []
def check(key, val):
    if val is None:
        missing.append(key)
        print(f'  ❌ MISSING: {key}')
    else:
        print(f'  ✅ {key}: {val.split("/")[-1]}')

for modality in ['txt', 'txtimg']:
    for lag in ['time_independent', 'lag1']:
        for dim in [128, 256]:
            check(f'{modality}/{lag}/{dim}/train', config[modality][lag][dim]['train'])
            check(f'{modality}/{lag}/{dim}/val',   config[modality][lag][dim]['val'])

for diff_key in ['diff_txt', 'diff_txtimg']:
    for lag in ['time_independent', 'lag1']:
        check(f'{diff_key}/{lag}/train', config[diff_key][lag]['train'])
        check(f'{diff_key}/{lag}/val',   config[diff_key][lag]['val'])

if missing:
    print(f'\n⚠️  {len(missing)} paths missing!')
else:
    print('\n✅ All 24 paths found correctly!')

=== Verifying all paths ===
  ✅ txt/time_independent/128/train: train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip
  ✅ txt/time_independent/128/val: val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip
  ✅ txt/time_independent/256/train: train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip
  ✅ txt/time_independent/256/val: val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip
  ✅ txt/lag1/128/train: train_pred_model-2026-05-31-embdim=None_lag1_txt_mod4-epoch=022-val_combined_loss=0.9805_txt_dim=128_proj_emb_step=.zip
  ✅ txt/lag1/128/val: val_pred_model-2026-05-31-embdim=None_lag1_txt_mod4-epoch=022-val_combined_loss=0.9805_txt_dim=128_proj_emb_step=.zip
  ✅ txt/lag1/256/train: 

## ④ Save paths_config.yaml

In [4]:
yaml_str = yaml.dump(config, default_flow_style=False, sort_keys=False)
print('=== paths_config.yaml ===')
print(yaml_str)

config_path = ROOT + 'code/utils/paths_config.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    f.write(yaml_str)
print(f'✅ Saved to: {config_path}')
print('\nNext step: re-run notebooks 01_1 and 01_2')

=== paths_config.yaml ===
txt:
  time_independent:
    128:
      train: ../data/predictions/txt/time_indipendent/train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip
      val: ../data/predictions/txt/time_indipendent/val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.8074_txt_dim=128_proj_emb_step=.zip
    256:
      train: ../data/predictions/txt/time_indipendent/train_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip
      val: ../data/predictions/txt/time_indipendent/val_pred_model-2026-05-31-embdim=None_time_independent_txt_mod4-epoch=018-val_combined_loss=1.6953_txt_dim=256_proj_emb_step=.zip
  lag1:
    128:
      train: ../data/predictions/txt/lag1/train_pred_model-2026-05-31-embdim=None_lag1_txt_mod4-epoch=022-val_combined_loss=0.9805_txt_dim=128_proj_emb_step=.zip
      val: ../da